# 01 · Bronze — Source extraction → CSV → DWH

Two-phase, VPN-aware extraction:
- **Phase A (VPN ON):** read source DB, save CSVs locally, also save column schema JSONs.
- **Phase B (VPN OFF):** read CSVs, derive bronze table DDL from the saved schema, diff against last snapshot, write to `bronze.*` and register the load.

Bronze is now **faithful to the source** — every column from each source table lands in bronze. Column projections happen later in silver.


## 1. Imports and env

In [2]:
import json
import pandas as pd
import hashlib
from datetime import datetime, date, timezone
from pathlib import Path
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

load_dotenv(override=True)

SRC_HOST     = os.getenv("SRC_HOST")
SRC_PORT     = int(os.getenv("SRC_PORT", 5432))
SRC_DB       = os.getenv("SRC_DB")
SRC_USER     = os.getenv("SRC_USER")
SRC_PASSWORD = os.getenv("SRC_PASSWORD")

SUPABASE_URL  = os.getenv("SUPABASE_URL")
SUPABASE_PORT = int(os.getenv("SUPABASE_PORT", 5432))
SUPABASE_DB   = os.getenv("SUPABASE_DB")
SUPABASE_USER = os.getenv("SUPABASE_USER")
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

def make_engine(host, port, db, user, password, sslmode=None):
    return create_engine(URL.create("postgresql+psycopg2", username=user, password=password, host=host, port=port, database=db, query={"sslmode": sslmode} if sslmode else None))

print("Env loaded. Create engine_src (VPN ON) or engine_dwh (VPN OFF) in the next cells.")


Env loaded. Create engine_src (VPN ON) or engine_dwh (VPN OFF) in the next cells.


## 2. engine_src — VPN ON

In [3]:
engine_src = make_engine(SRC_HOST, SRC_PORT, SRC_DB, SRC_USER, SRC_PASSWORD)
with engine_src.connect() as conn:
    print("✓ engine_src connected:", conn.execute(text("SELECT current_database()")).scalar())


✓ engine_src connected: wwi


## 3. engine_dwh — VPN OFF

In [6]:
if not all([SUPABASE_URL, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD]):
    raise ValueError("SUPABASE env vars missing")
engine_dwh = make_engine(SUPABASE_URL, SUPABASE_PORT, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD, sslmode="require")
with engine_dwh.connect() as conn:
    print("✓ engine_dwh connected:", conn.execute(text("SELECT current_database()")).scalar())


✓ engine_dwh connected: postgres


## 4. Table catalog

Snapshot tables are full-reloaded each run; date tables are appended incrementally based on a watermark column.

In [7]:
SNAPSHOT_TABLES = [
    {"src": "people", "dest": "people", "pk": ["personid"]},
    {"src": "customercategories", "dest": "customercategories", "pk": ["customercategoryid"]},
    {"src": "customers", "dest": "customers", "pk": ["customerid"]},
    {"src": "countries", "dest": "countries", "pk": ["countryid"]},
    {"src": "stateprovinces", "dest": "stateprovinces", "pk": ["stateprovinceid"]},
    {"src": "cities", "dest": "cities", "pk": ["cityid"]},
    {"src": "stockitems", "dest": "stockitems", "pk": ["stockitemid"]},
    {"src": "customertransactions", "dest": "customertransactions", "pk": ["customertransactionid"]},
    {"src": "deliverymethods", "dest": "deliverymethods", "pk": ["deliverymethodid"]},
    {"src": "paymentmethods", "dest": "paymentmethods", "pk": ["paymentmethodid"]},
    {"src": "transactiontypes", "dest": "transactiontypes", "pk": ["transactiontypeid"]},
]

DATE_TABLES = [
    {"src": "invoices", "dest": "invoices", "watermark": "invoicedate", "pk": ["invoiceid"]},
    {"src": "invoicelines", "dest": "invoicelines", "watermark": "invoicedate", "pk": ["invoicelineid"]},
]

SNAP_META_COLS = {
    "_snapshot_id": "INT NOT NULL",
    "_loaded_at": "TIMESTAMP NOT NULL",
    "_change_op": "VARCHAR(10) NOT NULL",
    "_source": "VARCHAR(100) NOT NULL",
}
DATE_META_COLS = {
    "_loaded_at": "TIMESTAMP NOT NULL",
    "_source": "VARCHAR(100) NOT NULL",
}


## 5. _load_control table (VPN OFF)

Already created by `00_setup.ipynb`. This cell ensures it exists if you skipped setup.

In [6]:
DDL_CONTROL = """
CREATE TABLE IF NOT EXISTS bronze._load_control (
    table_name VARCHAR(100) NOT NULL,
    strategy VARCHAR(20) NOT NULL,
    snapshot_id INT,
    watermark_date DATE,
    loaded_at TIMESTAMP NOT NULL,
    rows_total INT,
    rows_inserted INT,
    rows_updated INT,
    rows_deleted INT,
    status VARCHAR(20) NOT NULL,
    PRIMARY KEY (table_name, loaded_at)
);
"""
with engine_dwh.begin() as conn:
    conn.execute(text(DDL_CONTROL))
print("✓ bronze._load_control ready.")


✓ bronze._load_control ready.


## 6. Helper functions

All helpers below operate on `engine_dwh` only (VPN OFF). They handle:
- snapshot id allocation per table
- last-active-snapshot retrieval (excludes DELETE rows)
- row-level diff via MD5 hash
- load_control INSERT
- watermark retrieval for incremental tables
- DDL generation from a saved source schema JSON

In [9]:
PG_TYPE_MAP = {
    "integer": "INT", "bigint": "BIGINT", "smallint": "SMALLINT",
    "text": "TEXT", "character varying": "TEXT", "character": "TEXT",
    "date": "DATE", "timestamp without time zone": "TIMESTAMP", "timestamp with time zone": "TIMESTAMPTZ",
    "numeric": "NUMERIC", "real": "REAL", "double precision": "DOUBLE PRECISION",
    "boolean": "BOOLEAN", "bytea": "BYTEA", "uuid": "UUID", "json": "JSONB", "jsonb": "JSONB",
}

def map_pg_type(data_type):
    return PG_TYPE_MAP.get(data_type.lower(), "TEXT")

def ensure_bronze_table(dest, schema_json, meta_cols):
    """Create bronze.<dest> from a list of {column_name, data_type} entries plus the metadata columns."""
    cols_sql = []
    for entry in schema_json:
        cols_sql.append(f"{entry['column_name']} {map_pg_type(entry['data_type'])}")
    for col, ddl in meta_cols.items():
        cols_sql.append(f"{col} {ddl}")
    ddl = f"CREATE TABLE IF NOT EXISTS bronze.{dest} (" + ", ".join(cols_sql) + ");"
    with engine_dwh.begin() as conn:
        conn.execute(text(ddl))

def get_next_snapshot_id(table_name):
    with engine_dwh.connect() as conn:
        return conn.execute(text("SELECT COALESCE(MAX(snapshot_id), 0) + 1 FROM bronze._load_control WHERE table_name = :t"), {"t": table_name}).scalar()

def get_last_snapshot(table_name):
    try:
        with engine_dwh.connect() as conn:
            df = pd.read_sql(text(f"SELECT * FROM bronze.{table_name} WHERE _snapshot_id = (SELECT MAX(_snapshot_id) FROM bronze.{table_name}) AND _change_op != 'DELETE'"), conn)
        meta_cols = ["_snapshot_id", "_loaded_at", "_change_op", "_source"]
        return df.drop(columns=[c for c in meta_cols if c in df.columns])
    except Exception:
        return pd.DataFrame()

def get_watermark(table_name):
    """Read the last successful watermark_date for an incremental table. Defaults to 1900-01-01."""
    try:
        with engine_dwh.connect() as conn:
            res = conn.execute(text("SELECT MAX(watermark_date) FROM bronze._load_control WHERE table_name = :t AND status = 'SUCCESS' AND watermark_date IS NOT NULL"), {"t": table_name}).scalar()
        return res if res is not None else date(1900, 1, 1)
    except Exception:
        return date(1900, 1, 1)

def compute_row_hash(df, cols):
    return df[cols].astype(str).apply(lambda row: hashlib.md5("|".join(row).encode()).hexdigest(), axis=1)

def diff_snapshots(df_new, df_prev, pk):
    df_new = df_new.copy().set_index(pk)
    if df_prev.empty:
        res = df_new.reset_index()
        res["_change_op"] = "INSERT"
        return res
    df_prev = df_prev.copy().set_index(pk)
    common_cols = [c for c in df_new.columns if c in df_prev.columns]
    hash_new = compute_row_hash(df_new.reset_index(), pk + common_cols); hash_new.index = df_new.index
    hash_prev = compute_row_hash(df_prev.reset_index(), pk + common_cols); hash_prev.index = df_prev.index
    changed = []
    new_keys = df_new.index.difference(df_prev.index)
    if len(new_keys):
        ins = df_new.loc[new_keys].reset_index(); ins["_change_op"] = "INSERT"; changed.append(ins)
    common_keys = df_new.index.intersection(df_prev.index)
    updated_keys = common_keys[hash_new.loc[common_keys] != hash_prev.loc[common_keys]]
    if len(updated_keys):
        upd = df_new.loc[updated_keys].reset_index(); upd["_change_op"] = "UPDATE"; changed.append(upd)
    deleted_keys = df_prev.index.difference(df_new.index)
    if len(deleted_keys):
        dlt = df_prev.loc[deleted_keys].reset_index().reindex(columns=df_new.reset_index().columns)
        dlt["_change_op"] = "DELETE"; changed.append(dlt)
    return pd.concat(changed, ignore_index=True) if changed else pd.DataFrame()

def register_load(table_name, strategy, loaded_at, snapshot_id=None, watermark_date=None, rows_total=0, rows_inserted=0, rows_updated=0, rows_deleted=0, status="SUCCESS"):
    with engine_dwh.begin() as conn:
        conn.execute(text("""
            INSERT INTO bronze._load_control (table_name, strategy, snapshot_id, watermark_date, loaded_at, rows_total, rows_inserted, rows_updated, rows_deleted, status)
            VALUES (:tname, :strategy, :sid, :wm_date, :ts, :total, :inserted, :updated, :deleted, :status)
        """), {"tname": table_name, "strategy": strategy, "sid": snapshot_id, "wm_date": watermark_date, "ts": loaded_at, "total": rows_total, "inserted": rows_inserted, "updated": rows_updated, "deleted": rows_deleted, "status": status})


## 7. Save snapshots to local CSV (VPN ON)

For each snapshot table: dump every column and write `tmp_snapshots/<table>.csv` plus the column-schema JSON in `tmp_snapshots/_schema/<table>.json`.

In [9]:
tmp_dir = Path("tmp_snapshots"); tmp_dir.mkdir(exist_ok=True)
schema_dir = tmp_dir / "_schema"; schema_dir.mkdir(exist_ok=True)
run_at = datetime.now(timezone.utc).replace(tzinfo=None)
if 'engine_src' not in globals() or engine_src is None:
    raise RuntimeError("engine_src missing — run cell '## 2. engine_src' (VPN ON).")

print("Saving source snapshots ...")
for t in SNAPSHOT_TABLES:
    src, dest = t["src"], t["dest"]
    try:
        with engine_src.connect() as conn:
            schema_df = pd.read_sql(text("""
                SELECT column_name, data_type, is_nullable
                FROM information_schema.columns
                WHERE table_schema='public' AND table_name=:t
                ORDER BY ordinal_position
            """), conn, params={"t": src})
        if schema_df.empty:
            print(f"⚠ {src}: source table not found, skipping.")
            continue
        schema_entries = schema_df.to_dict(orient="records")
        (schema_dir / f"{dest}.json").write_text(json.dumps(schema_entries, indent=2), encoding="utf-8")
        cols_str = ", ".join([e["column_name"] for e in schema_entries])
        with engine_src.connect() as conn:
            df_new = pd.read_sql(text(f"SELECT {cols_str} FROM public.{src}"), conn)
        df_new.to_csv(tmp_dir / f"{dest}.csv", index=False)
        print(f"  {dest}: {len(df_new)} rows, {len(schema_entries)} cols → {tmp_dir / f'{dest}.csv'}")
    except Exception as e:
        print(f"  ✗ {dest}: {e}")


Saving source snapshots ...
  people: 1111 rows, 18 cols → tmp_snapshots\people.csv
  customercategories: 8 rows, 2 cols → tmp_snapshots\customercategories.csv
  customers: 663 rows, 28 cols → tmp_snapshots\customers.csv
  countries: 190 rows, 8 cols → tmp_snapshots\countries.csv
  stateprovinces: 53 rows, 6 cols → tmp_snapshots\stateprovinces.csv
  cities: 37940 rows, 5 cols → tmp_snapshots\cities.csv
  stockitems: 227 rows, 22 cols → tmp_snapshots\stockitems.csv
  customertransactions: 97147 rows, 12 cols → tmp_snapshots\customertransactions.csv
  deliverymethods: 10 rows, 2 cols → tmp_snapshots\deliverymethods.csv
  paymentmethods: 4 rows, 2 cols → tmp_snapshots\paymentmethods.csv
  transactiontypes: 13 rows, 2 cols → tmp_snapshots\transactiontypes.csv


## 8. Apply snapshots to DWH (VPN OFF)

For each table: read CSV + schema JSON, ensure bronze table exists with correct DDL, diff against previous snapshot, append diff with metadata, register in `_load_control`.

In [10]:
tmp_dir = Path("tmp_snapshots"); schema_dir = tmp_dir / "_schema"
run_at = datetime.now(timezone.utc).replace(tzinfo=None)
if 'engine_dwh' not in globals() or engine_dwh is None:
    raise RuntimeError("engine_dwh missing — run cell '## 3. engine_dwh' (VPN OFF).")

print("Applying snapshots to DWH...")
for t in SNAPSHOT_TABLES:
    dest, pk = t["dest"], t["pk"]
    csv_path = tmp_dir / f"{dest}.csv"
    schema_path = schema_dir / f"{dest}.json"
    if not csv_path.exists() or not schema_path.exists():
        print(f"  · {dest}: missing CSV or schema, skipping.")
        continue
    try:
        schema_entries = json.loads(schema_path.read_text(encoding="utf-8"))
        ensure_bronze_table(dest, schema_entries, SNAP_META_COLS)
        df_new = pd.read_csv(csv_path)
        df_prev = get_last_snapshot(dest)
        df_diff = diff_snapshots(df_new, df_prev, pk)
        n_total = len(df_new)
        if df_diff.empty:
            register_load(dest, "snapshot", run_at, rows_total=n_total)
            print(f"  · {dest}: no changes ({n_total} rows).")
            continue
        snap_id = get_next_snapshot_id(dest)
        df_diff["_snapshot_id"] = snap_id
        df_diff["_loaded_at"] = run_at
        df_diff["_source"] = SRC_DB
        with engine_dwh.connect() as conn:
            target_cols = pd.read_sql(text("SELECT column_name FROM information_schema.columns WHERE table_schema='bronze' AND table_name=:t"), conn, params={"t": dest})["column_name"].tolist()
        df_diff = df_diff[[c for c in df_diff.columns if c in target_cols]]
        with engine_dwh.begin() as conn:
            df_diff.to_sql(dest, conn, schema="bronze", if_exists="append", index=False)
        i = int((df_diff["_change_op"] == "INSERT").sum())
        u = int((df_diff["_change_op"] == "UPDATE").sum())
        d = int((df_diff["_change_op"] == "DELETE").sum())
        register_load(dest, "snapshot", run_at, snapshot_id=snap_id, rows_total=n_total, rows_inserted=i, rows_updated=u, rows_deleted=d)
        print(f"  · {dest}: ins={i}, upd={u}, del={d} (snap {snap_id}).")
    except Exception as e:
        register_load(dest, "snapshot", run_at, status="ERROR")
        print(f"  ✗ {dest}: {e}")


Applying snapshots to DWH...
  · people: ins=1111, upd=0, del=0 (snap 1).
  · customercategories: ins=8, upd=0, del=0 (snap 1).
  · customers: ins=663, upd=0, del=0 (snap 1).
  · countries: ins=190, upd=0, del=0 (snap 1).
  · stateprovinces: ins=53, upd=0, del=0 (snap 1).
  · cities: ins=37940, upd=0, del=0 (snap 1).
  · stockitems: ins=227, upd=0, del=0 (snap 1).
  · customertransactions: ins=97147, upd=0, del=0 (snap 1).
  · deliverymethods: ins=10, upd=0, del=0 (snap 1).
  · paymentmethods: ins=4, upd=0, del=0 (snap 1).
  · transactiontypes: ins=13, upd=0, del=0 (snap 1).


## 9. Save incremental extracts (VPN ON)

Reads `bronze._load_control` watermark first (this needs the DWH, so we **also** open `engine_dwh` here for read-only access — VPN OFF). If you cannot reach the DWH right now, the watermark falls back to `1900-01-01` and the extract becomes a full load.

**Recommended flow:** run this cell with VPN OFF first to capture the watermark in a Python variable, then run with VPN ON to extract.

In [4]:
tmp_incr = Path("tmp_increments"); tmp_incr.mkdir(exist_ok=True)
schema_dir_incr = tmp_incr / "_schema"; schema_dir_incr.mkdir(exist_ok=True)
run_at = datetime.now(timezone.utc).replace(tzinfo=None)

if 'engine_src' not in globals() or engine_src is None:
    raise RuntimeError("engine_src missing — run cell '## 2. engine_src' (VPN ON).")

wm_invoices = get_watermark("invoices") if 'engine_dwh' in globals() and engine_dwh is not None else date(1900, 1, 1)
print(f"Watermark for invoices: {wm_invoices}")

# 1) Source schema for invoices
with engine_src.connect() as conn:
    inv_schema = pd.read_sql(text("SELECT column_name, data_type, is_nullable FROM information_schema.columns WHERE table_schema='public' AND table_name='invoices' ORDER BY ordinal_position"), conn).to_dict(orient="records")
    line_schema = pd.read_sql(text("SELECT column_name, data_type, is_nullable FROM information_schema.columns WHERE table_schema='public' AND table_name='invoicelines' ORDER BY ordinal_position"), conn).to_dict(orient="records")
(schema_dir_incr / "invoices.json").write_text(json.dumps(inv_schema, indent=2), encoding="utf-8")
(schema_dir_incr / "invoicelines.json").write_text(json.dumps(line_schema, indent=2), encoding="utf-8")

# 2) Invoices: full columns since watermark
inv_cols = ", ".join([e["column_name"] for e in inv_schema])
with engine_src.connect() as conn:
    df_inv = pd.read_sql(text(f"SELECT {inv_cols} FROM public.invoices WHERE invoicedate > :wm ORDER BY invoicedate"), conn, params={"wm": wm_invoices})
df_inv.to_csv(tmp_incr / "invoices.csv", index=False)
print(f"  invoices: {len(df_inv)} rows since {wm_invoices}")

# 3) Invoicelines for the new invoices
if not df_inv.empty:
    line_cols = ", ".join([e["column_name"] for e in line_schema])
    new_ids = df_inv["invoiceid"].tolist()
    with engine_src.connect() as conn:
        df_lines = pd.read_sql(text(f"SELECT {line_cols} FROM public.invoicelines WHERE invoiceid = ANY(:ids)"), conn, params={"ids": new_ids})
    df_lines.to_csv(tmp_incr / "invoicelines.csv", index=False)
    print(f"  invoicelines: {len(df_lines)} rows")
else:
    pd.DataFrame().to_csv(tmp_incr / "invoicelines.csv", index=False)
    print("  invoicelines: 0 rows (no new invoices)")


Watermark for invoices: 1900-01-01
  invoices: 70510 rows since 1900-01-01
  invoicelines: 228265 rows


## 10. Apply incremental CSVs to DWH (VPN OFF)

In [10]:
tmp_incr = Path("tmp_increments"); schema_dir_incr = tmp_incr / "_schema"
run_at = datetime.now(timezone.utc).replace(tzinfo=None)
if 'engine_dwh' not in globals() or engine_dwh is None:
    raise RuntimeError("engine_dwh missing — run cell '## 3. engine_dwh' (VPN OFF).")

print("Applying incremental CSVs to DWH...")
for t in DATE_TABLES:
    dest, pk = t["dest"], t["pk"]
    csv_path = tmp_incr / f"{dest}.csv"
    schema_path = schema_dir_incr / f"{dest}.json"
    if not csv_path.exists() or not schema_path.exists():
        print(f"  · {dest}: missing CSV or schema, skipping.")
        continue
    try:
        schema_entries = json.loads(schema_path.read_text(encoding="utf-8"))
        ensure_bronze_table(dest, schema_entries, DATE_META_COLS)
        df_new = pd.read_csv(csv_path)
        if df_new.empty:
            register_load(dest, "incremental", run_at, rows_total=0)
            print(f"  · {dest}: empty CSV, nothing to load.")
            continue
        df_new["_loaded_at"] = run_at
        df_new["_source"] = SRC_DB
        with engine_dwh.connect() as conn:
            target_cols = pd.read_sql(text("SELECT column_name FROM information_schema.columns WHERE table_schema='bronze' AND table_name=:t"), conn, params={"t": dest})["column_name"].tolist()
        df_to_load = df_new[[c for c in df_new.columns if c in target_cols]]
        with engine_dwh.begin() as conn:
            df_to_load.to_sql(dest, conn, schema="bronze", if_exists="append", index=False)
        wm_date = None
        if t["watermark"] in df_new.columns:
            try:
                wm_date = pd.to_datetime(df_new[t["watermark"]]).max().date()
            except Exception:
                wm_date = None
        register_load(dest, "incremental", run_at, watermark_date=wm_date, rows_total=len(df_new), rows_inserted=len(df_new))
        print(f"  · {dest}: {len(df_new)} rows loaded (watermark: {wm_date}).")
    except Exception as e:
        register_load(dest, "incremental", run_at, status="ERROR")
        print(f"  ✗ {dest}: {e}")


Applying incremental CSVs to DWH...
  · invoices: 70510 rows loaded (watermark: 2020-05-31).
  · invoicelines: 228265 rows loaded (watermark: None).
